[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/projective_loop/projective_loop.ipynb)

# Loop Subdivision of Weighted Points

This example computes Loop subdivision of a triangle mesh whose vertices are weighted points. Loop subdivision smooths a mesh: every triangle splits in four, every vertex moves to an average of itself and its neighbours, every edge gets a new vertex between its ends, and repeated steps converge to a smooth surface. Here the vertices are points of projective space, each with its own weight, and the averages are taken of the points as they are, weights included. Two things follow. A vertex's weight is a handle on the surface: a heavy vertex pulls the surface towards itself, a light one lets it go. And subdivision commutes with perspective: refining and then viewing in perspective gives what viewing in perspective and then refining gives, where averaging positions does not.

The vertices are points of projective geometric algebra, and one Loop step is a map from the coarser mesh's vertex sites to the finer one's, `step * controls`, and the steps compose into one map. A perspective is a map on points, the identity plus one term, `Point + centre * (axis & Point) * strength`. The step acts across the sites and the perspective on the point at each site, so the two commute.

In the homogeneous-coordinate notation of computer-aided geometric design, each weighted vertex reads as $(w\,\mathbf p,\ w)$, a Loop step reads as a subdivision matrix $S$ acting across the vertices, and a projective transformation reads as a $4 \times 4$ homography matrix $H$ acting on each vertex, so rational Loop subdivision satisfies $(S V) H^{\mathsf T} = S (V H^{\mathsf T})$ with weights acting as in rational Bézier patches and NURBS.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import PGA3D as ga
from examples import instantiate
from examples.util.animation import save_figure, show_animation
from examples.surfaces.projective_loop import render

mv = NumpyContext(ga).multivector
Point = ga.gatype.antivector()                      # Antivector-
Mesh = instantiate("examples.util.mesh", ga).Mesh

# The octahedron's six vertices, with `top_vertex` at the +z apex, and its eight counter-clockwise faces.
octahedron = np.array([[1, 0, 0], [-1, 0, 0], [0, 1, 0], [0, -1, 0], [0, 0, 1], [0, 0, -1]])
top_vertex = 4
octahedron_faces = np.array([[0, 2, 4], [2, 1, 4], [1, 3, 4], [3, 0, 4], [2, 0, 5], [1, 2, 5], [3, 1, 5], [0, 3, 5]])

## 1. A Loop step as a map on the vertex sites

In one Loop step, an existing vertex of valence $n$ retains $1 - n\beta$ of itself and takes $\beta$ of each neighbour, with $\beta = \tfrac1n\big(\tfrac58 - (\tfrac38 + \tfrac14\cos\tfrac{2\pi}{n})^2\big)$, while each new edge vertex takes $\tfrac38$ of each endpoint and $\tfrac18$ of each opposite corner across the edge. Because these shares depend only on how the mesh is connected, each step is a linear map from the vertex sites of the coarse mesh to those of the fine one, the existing vertices followed by one per edge. It acts on whatever sits at the sites, here points with their weights, and composing three steps produces one map from the octahedron's six vertex sites to the 258 of the refined mesh.

In [ ]:
# Number of Loop subdivision steps.
n_loop_steps = 3

In [ ]:
# Three Loop steps of the octahedron, none of its vertices pinned. The refined mesh holds the steps, composed into
# one map from the octahedron's vertex sites to its own.
refined_mesh = Mesh(len(octahedron), octahedron_faces).loop_subdivided(np.zeros(len(octahedron), dtype=bool), n_loop_steps)
subdivision = refined_mesh.subdivision                                          # Scalar+[refined <- vertices]
print(subdivision.gatype, subdivision.shape)

## 2. Vertex weights as handles on the surface

The same steps refine the octahedron with all weights alike, with its top vertex four times as heavy, and with it four times as light, all three as one batch. With equal weights the surface is ordinary Loop subdivision of the positions. The heavy top vertex draws the surface up towards it; the light one lets it sink back towards the middle. The dotted lines are the control mesh. The animation sweeps the top weight from a quarter to four and back, each frame's octahedron through the one composed map.

In [ ]:
# Vertex weights for the three static cases (uniform, heavy top vertex, light top vertex) and parameters
# for the log-cosine weight sweep animation.
weights = np.array([[1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 4.0, 1.0], [1.0, 1.0, 1.0, 1.0, 0.25, 1.0]])
weight_bounds = (0.25, 4.0)
n_sweep_frames = 48
duration_ms = 60

In [ ]:
controls = (mv.point(octahedron) * weights).field()                            # [cases] Point-[vertices]
surfaces = subdivision * controls                                              # [cases] Point-[refined]

save_figure(render.draw_weights(controls, octahedron_faces, surfaces, refined_mesh.faces), "projective_loop_weights");

In [ ]:
# Sweep the top weight from light to heavy along a cosine in log-weight; the animation plays forward and back.
phase = np.linspace(0, np.pi, n_sweep_frames // 2 + 1)
top_weight = np.exp(np.log(weight_bounds).mean() - np.log(weight_bounds[1] / weight_bounds[0]) / 2 * np.cos(phase))  # [frames]
sweep_weights = np.where(np.arange(len(octahedron)) == top_vertex, top_weight[:, None], 1.0)  # [frames, vertices]
sweep_controls = ((mv.point(octahedron) * weight).field() for weight in sweep_weights)

show_animation(render.animate_weights(((control, subdivision * control) for control in sweep_controls), octahedron_faces, refined_mesh.faces),
               "projective_loop_sweep", duration_ms, forward_then_back=True)

## 3. Commuting with a perspective collineation

A perspective here is a central collineation: each point moves along its line through a centre, by the centre times its pairing with an axis plane, which leaves the axis in place and changes the weight of every point off it. It is the identity plus one more term, `Point + centre * (axis & Point) * strength`. The Loop step acts across the vertex sites and the collineation on the point at each site, so the two commute: refining and then viewing equals viewing and then refining. Averaging positions instead divides each point by its weight first, and the collineation changes the weights, so there the two orders part. Each surface is coloured by how far the other order's surface lies, on one scale for both panels: with weights the two agree everywhere, with positions they part most where the perspective changes the weights most.

In matrix notation, with the homogeneous vertex coordinates arranged as the rows of $V$, the subdivision operator reads as a matrix $S$ acting from the left and the collineation reads as a $4 \times 4$ matrix $H$ acting from the right, so $S (V H^{\mathsf T}) = (S V) H^{\mathsf T}$, whereas normalising rows to unit weight inserts a diagonal rescaling matrix that does not commute with $S$.

In [ ]:
# Stretch and offset placing the octahedron in front of a central collineation with centre at the origin
# and axis plane z = 0.
stretch = np.array([1.0, 1.0, 1.5])
offset = np.array([0.0, 0.0, 1.6])
centre, axis, strength = mv.w.dual(), mv.z, 1.0

In [ ]:
rest_control = mv.point(octahedron * stretch + offset).field()                 # [] Point-[vertices]
# The central collineation shifts each point along its line through the centre in proportion to its signed
# distance from the axis plane.
collineation = Point + centre * (axis & Point) * strength                      # [] Point- <- Point-
perspective_control = collineation(rest_control)                               # [] Point-[vertices]
weighted_orders = stack([subdivision * perspective_control, collineation(subdivision * rest_control)])  # [orders] Point-[refined]
rest_positions = rest_control.normalized()                                      # [] Point-[vertices]
perspective_positions = perspective_control.normalized()                        # [] Point-[vertices]
positional_orders = stack([subdivision * perspective_positions, collineation(subdivision * rest_positions)])  # [orders] Point-[refined]
# Normalising each refined surface to unit weight makes the norm of the join between the two orders equal
# to their Euclidean separation.
weighted_unit = weighted_orders.normalized()                                    # [orders] Point-[refined]
positional_unit = positional_orders.normalized()                                # [orders] Point-[refined]
weighted_gaps = (weighted_unit[0] & weighted_unit[1]).norm()                   # [] Scalar+[refined]
positional_gaps = (positional_unit[0] & positional_unit[1]).norm()             # [] Scalar+[refined]

print(f"largest gap between the orders of Euclidean subdivision: {positional_gaps.to_array().max():.3f}")
save_figure(render.draw_orders(perspective_control, octahedron_faces, weighted_orders, positional_orders, refined_mesh.faces),
            "projective_loop_perspective");

In [ ]:
# checks
assert np.all(weighted_gaps < 1e-12)
assert np.any(positional_gaps > 1e-2)
# With all weights equal, weighted projective subdivision agrees with Euclidean position subdivision.
assert np.all((surfaces[0] & surfaces[0].normalized()).norm() < 1e-12)